# Converting BSI output to tidy format for graphing

**SUMMARY:** 

This notebook **BSI_to_tidy** takes a **Blind_Planarian_Scoring_[Name].csv** for data and **anonymization_lookup.csv** to fill in a **Tidy_template.csv**. The first csv contains the scoring data of anonymous videos from the Blind Scoring Interface (a human creates this by using the interface). The anonymization lookup csv was made automatically when raw videos were given hidden names using LF Notebook 2; it reveals what hidden name corresponds to what raw video and which worm in that video. The tidy template is structured in the way that the graphing code expects headers, rows, etc., and it gets filled in by information the scoring csv and the anonymization csv. The output of this notebook is fed into the Tasmanian graphing code, either independently or as part of the interscorer comparisons.


The BSI CSV stores data with the columns:
    
    Video_filename, TimestampID, Number_turns. Turn_timestamps, Number_contractions, Contraction_timestamps, Notes

The manual scoring sheet stores data with the columns:
   
    Run, Troupe, Day, Block, Trial, CRturn_W1, ..., CRturn_W6 CRcon_W1, ..., CRconW6, UCRturnW1, ..., UCRturnW6, UCRconW1, ..., UCRconW6, Notes


The anonymization lookup sheet stores data with these columns:

    Anonymous_Name, Original_File, Box_Coordinates, Trial_Number, Trial_Frames, Session_Date, Trial_Type, Timestamp_Created


## Imports, load CSV files, defining functions, and executing.

In [4]:
import pandas as pd
import numpy as np

# File paths
# BSI_DATA = "/Users/zacharykelso/Desktop/SAGE_Blind_Planarian_Scoring_Combined_20251204_112624.csv"
# BSI_DATA = "/Users/zacharykelso/Desktop/Blind_Planarian_Scoring_Combined_20260115_213302.csv" # Sage's BSI data
# BSI_DATA = "/Users/zacharykelso/Desktop/Blind_Planarian_Scoring_Fionn_20260127_1528.csv" # Fionn's data


BSI_DATA = "/Users/zacharykelso/Desktop/Zach_Tasmdata_untransf.csv"   # Enter path to csv generated by BSI here
LOOKUP_CSV = "/Users/zacharykelso/Desktop/anonymization_lookup.csv"   # Enter path to anonymization_lookup.csv here
CONVERTED_DATA = "/Users/zacharykelso/Desktop/Tidy_template.csv"      # Enter the template for the transforme data here

def fill_empty_run_cells(df):
    """
    Fill empty cells in 'Run' column by copying from the previous non-empty cell.
    """
    df['Run'] = df['Run'].ffill()
    return df

def find_timestamp_worm_mapping(lookup_df):
    """
    Create a mapping of TimestampID to worm identities (W1-W6) based on box coordinates.
    For each TimestampID, there should be exactly 6 unique box coordinates (one per worm).
    Returns a dict: {TimestampID: {box_coordinate: worm_number}}
    """
    timestamp_worm_map = {}
    
    # Get unique timestamp IDs
    unique_timestamps = lookup_df['TimestampID'].unique()
    print(f"     Found {len(unique_timestamps)} unique TimestampIDs")
    
    for timestamp_id in unique_timestamps:
        # Get all rows for this timestamp
        timestamp_rows = lookup_df[lookup_df['TimestampID'] == timestamp_id]
        
        # Extract UNIQUE box coordinates for this TimestampID
        box_coords = timestamp_rows['Box_Coordinates'].unique().tolist()
        
        print(f"\n     Checking TimestampID: '{timestamp_id}'")
        print(f"        Found {len(timestamp_rows)} total rows (across all trials)")
        print(f"        Found {len(box_coords)} unique box coordinates")
        
        if len(box_coords) != 6:
            print(f"         WARNING: Expected exactly 6 box coordinates, found {len(box_coords)}")
            print(f"        ALL box coordinates found:")
            for i, coord in enumerate(sorted(box_coords), 1):
                count = (timestamp_rows['Box_Coordinates'] == coord).sum()
                print(f"         {i}. {coord} (appears {count} times)")
        
        # Sort by the first number (X-coordinate, before first underscore)
        def get_first_number(coord_string):
            return int(str(coord_string).split('_')[0])
        
        sorted_coords = sorted(box_coords, key=get_first_number)
        
        # Create mapping: box_coordinate -> worm number (W1, W2, etc.)
        coord_to_worm = {}
        print(f"        Worm assignments (sorted by X-coordinate):")
        for idx, coord in enumerate(sorted_coords):
            worm_number = idx + 1  # W1 = 1, W2 = 2, etc.
            coord_to_worm[coord] = worm_number
            print(f"         W{worm_number} = {coord}")
        
        timestamp_worm_map[timestamp_id] = coord_to_worm
    
    return timestamp_worm_map

def convert_to_manual_format():
    """
    Main function to convert BSI data to manual scoring format.
    """
    # Read all CSVs
    print("\n" + "="*25)
    print("STEP 1: Reading CSV files...")
    print("="*25)
    
    bsi_df = pd.read_csv(BSI_DATA)
    print(f"  BSI data loaded: {len(bsi_df)} rows, {len(bsi_df.columns)} columns")
    
    # CLEAN VIDEO FILENAMES: Remove trailing slashes
    bsi_df['Video_filename'] = bsi_df['Video_filename'].str.rstrip('/')
    
    lookup_df = pd.read_csv(LOOKUP_CSV)
    print(f"  Lookup data loaded: {len(lookup_df)} rows, {len(lookup_df.columns)} columns")
    
    # CLEAN LOOKUP DATA: Strip whitespace from TimestampID
    print(f"    Cleaning TimestampID column...")
    lookup_df['TimestampID'] = lookup_df['TimestampID'].str.strip()
    
    converted_df = pd.read_csv(CONVERTED_DATA)
    print(f"  Template data loaded: {len(converted_df)} rows")
    
    # Step 1a: Fill empty Run cells
    print("\n" + "="*25)
    print("STEP 2: Filling empty Run cells...")
    print("="*25)
    converted_df = fill_empty_run_cells(converted_df)
    print(f"  Run column filled")
    
    # Step 1b: Create timestamp to worm mapping
    print("\n" + "="*25)
    print("STEP 3: Creating timestamp-worm mapping...")
    print("="*25)
    timestamp_worm_map = find_timestamp_worm_mapping(lookup_df)
    print(f"\n  Mapping created for {len(timestamp_worm_map)} timestamps")
    
    # Step 1b.iii: Process each BSI entry
    print("\n" + "="*25)
    print("STEP 4: Processing BSI data entries...")
    print("="*25)
    
    # Statistics trackers
    successful_matches = 0
    no_lookup_match = 0
    no_converted_match = 0
    skipped_headers = 0
    
    # Track which videos don't match
    videos_no_lookup = []
    videos_no_template = []
    videos_skipped_headers = []
    
    for idx, bsi_row in bsi_df.iterrows():
        video_filename = bsi_row['Video_filename']
        
        # Skip if it looks like a header row
        if video_filename == 'Video_filename':
            print(f"  Skipping header row at index {idx}")
            skipped_headers += 1
            videos_skipped_headers.append(f"Row {idx} (duplicate header)")
            continue
        
        # Find matching row in lookup CSV
        lookup_match = lookup_df[lookup_df['Anonymous_Name'] == video_filename]
        
        if lookup_match.empty:
            print(f"  No lookup match for: '{video_filename}'")
            no_lookup_match += 1
            videos_no_lookup.append(video_filename)
            continue
        
        # Get the first match (should only be one)
        lookup_row = lookup_match.iloc[0]
        
        # Use TimestampID for the Run name
        run_name = lookup_row['TimestampID'].strip()  # Strip whitespace
        box_coordinates = lookup_row['Box_Coordinates']
        trial_number = lookup_row['Trial_Number']
        timestamp_id = lookup_row['TimestampID'].strip()  # Strip whitespace
        
        # Find matching row in converted data
        converted_match = converted_df[
            (converted_df['Run'] == run_name) & 
            (converted_df['Trial'] == trial_number)
        ]
        
        if converted_match.empty:
            print(f"  No template match: {video_filename}")
            print(f"    Run={run_name}, Trial={trial_number}")
            no_converted_match += 1
            videos_no_template.append(video_filename)
            continue
        
        # Get the index of the matching row
        converted_idx = converted_match.index[0]
        
        # Determine worm number (W1-W6)
        if box_coordinates not in timestamp_worm_map[timestamp_id]:
            print(f"  ERROR: Box coordinate {box_coordinates} not found in mapping for {timestamp_id}")
            no_converted_match += 1
            videos_no_template.append(video_filename)
            continue
            
        worm_num = timestamp_worm_map[timestamp_id][box_coordinates]
        
        # Fill in the data
        cr_turn_col = f'CRturn_W{worm_num}'
        cr_con_col = f'CRcon_W{worm_num}'
        
        # Convert to int to avoid FutureWarning
        converted_df.at[converted_idx, cr_turn_col] = int(bsi_row['Number_turns'])
        converted_df.at[converted_idx, cr_con_col] = int(bsi_row['Number_contractions'])
        
        # Append notes if they exist
        if pd.notna(bsi_row['Notes']) and bsi_row['Notes'] != '':
            existing_notes = converted_df.at[converted_idx, 'Notes']
            note_to_add = f"W{worm_num}: {bsi_row['Notes']}"
            
            if pd.isna(existing_notes) or existing_notes == '':
                converted_df.at[converted_idx, 'Notes'] = str(note_to_add)
            else:
                converted_df.at[converted_idx, 'Notes'] = f"{existing_notes}; {note_to_add}"
        
        print(f"  {video_filename} → W{worm_num} (T:{bsi_row['Number_turns']}, C:{bsi_row['Number_contractions']})")
        successful_matches += 1
    
    # Print summary statistics
    print("\n" + "="*25)
    print("STEP 5: Summary Statistics")
    print("="*25)
    total_entries = len(bsi_df)
    print(f"Total BSI entries processed: {total_entries}")
    print(f"No. successful matches: {successful_matches} ({successful_matches/total_entries*100:.1f}%)")
    print(f"No. skipped header rows: {skipped_headers} ({skipped_headers/total_entries*100:.1f}%)")
    print(f"No. no lookup matches: {no_lookup_match} ({no_lookup_match/total_entries*100:.1f}%)")
    print(f"No. no template matches: {no_converted_match} ({no_converted_match/total_entries*100:.1f}%)")
    
    # Print lists of unmatched videos
    if videos_skipped_headers:
        print(f"\nSkipped header rows:")
        for vid in videos_skipped_headers:
            print(f"  - {vid}")
    
    if videos_no_lookup:
        print(f"\nVideos with NO LOOKUP MATCH:")
        for vid in videos_no_lookup:
            print(f"  - {vid}")
    else:
        print(f"\nYAY! All video files found in lookup table.")
    
    if videos_no_template:
        print(f"\nVideos with NO TEMPLATE MATCH:")
        for vid in videos_no_template:
            print(f"  - {vid}")
    else:
        print(f"\nYAY! All videos matched to template rows.")
    
    print("\n" + "="*25)
    
    # Save the result
    output_path = CONVERTED_DATA.replace('.csv', '_FILLED.csv')
    converted_df.to_csv(output_path, index=False)
    
    print(f"  CONVERSION COMPLETE!")
    print(f"  Output saved to: {output_path}")
    print(f"\nIt is recommended that you manually rename the file to the format: [Name]_Tasmdata_Tidy.csv")
    print("="*25 + "\n")
    
    return converted_df

# Run the conversion
if __name__ == "__main__":
    result_df = convert_to_manual_format()


STEP 1: Reading CSV files...
  BSI data loaded: 216 rows, 7 columns
  Lookup data loaded: 427 rows, 8 columns
    Cleaning TimestampID column...
  Template data loaded: 288 rows

STEP 2: Filling empty Run cells...
  Run column filled

STEP 3: Creating timestamp-worm mapping...
     Found 20 unique TimestampIDs

     Checking TimestampID: '2025_09_19_13_53_26_trial_1_TC'
        Found 157 total rows (across all trials)
        Found 13 unique box coordinates
        ALL box coordinates found:
         1. 1071_501_1324_1469 (appears 6 times)
         2. 108_522_371_1442 (appears 6 times)
         3. 1093_522_1314_1458 (appears 15 times)
         4. 1326_495_1568_1463 (appears 6 times)
         5. 1332_543_1556_1466 (appears 15 times)
         6. 138_519_359_1445 (appears 15 times)
         7. 366_509_605_1455 (appears 6 times)
         8. 377_535_604_1450 (appears 15 times)
         9. 542_192_1093_436 (appears 31 times)
         10. 592_514_845_1458 (appears 6 times)
         11. 622_5

/var/folders/87/dpmgsfdn41g25q40htwmw6nr0000gq/T/ipykernel_54091/415386083.py:191: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'W2: Shaking of trough during trial' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  converted_df.at[converted_idx, 'Notes'] = str(note_to_add)
